In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-stage-3-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Transforms
train_transform = transforms.Compose([
    # Use RandomRotation(15) Augmentation on the training dataset + Reize the images to 32x32
    transforms.RandomRotation(15),
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])


# Important Note: in my laptop when i load the data it's go to different path not the normal way
# and i try to fix it and asked some TA's to solve the problem but they don't know how because its first time to them see the problem
# to be clear path is supposed to be as first cell but when i run the cell it goes here Path to dataset files: /kaggle/input/q1-stage-3-2026
# so i solve the q2 by using my path to ensrue everything is correct the n i use the normal way that most students have
# i hope you got my point :)
# Datasets and DataLoaders
train_dir = os.path.join(path, "PotatoDisease", "train")
test_dir  = os.path.join(path, "PotatoDisease", "val")

train_dataset = datasets.ImageFolder(root=train_dir, transform=train_transform)
test_dataset  = datasets.ImageFolder(root=test_dir,  transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

# Display some sample images with labels
images, labels = next(iter(train_loader))
classes = train_dataset.classes

plt.figure(figsize=(8, 4))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    img = images[i].permute(1, 2, 0)
    plt.imshow(img.numpy())
    plt.axis('off')
    plt.title(classes[labels[i].item()])
plt.tight_layout()
plt.show()


In [ ]:
# Write your code here
import torch.nn as nn

class PotatoCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x



In [ ]:
# Write your code here
import torch

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy


def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy



In [ ]:
# Write your code here
import torch.optim as optim

# Set up device, model, loss function, and optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = PotatoCNN(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train the model
num_epochs = 10
train_losses = []
val_losses= []
train_accuracies=[]
val_accuracies = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}%, "
          f"Val Loss={val_loss:.4f}, Val Acc={val_acc:.2f}%")

#Plot the training and validation losses and training and validation accuracy

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker="o")
plt.plot(range(1, num_epochs+1), val_losses, label="Val Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()


plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Acc", marker="o")
plt.plot(range(1, num_epochs+1), val_accuracies, label="Val Acc", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# Write your code here
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Redefine the CNN model but add a residual (skip) connection from the 2nd convolutional layer to the 4th convolutional layer
class PotatoCNNResidual(nn.Module):
    def __init__(self, num_classes=3):
        super(PotatoCNNResidual, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True)
        )

        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True)
        )

        self.conv4 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True)
        )

        # projecton the second layer channel (32) to  the 4th layer channels (128)
        self.skip_proj = nn.Conv2d(32, 128, kernel_size=1)

        self.conv5 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def forward(self, x):
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.conv4(x3)

        skip = self.skip_proj(x2)    # project 2nd conv output
        x4 = x4 + skip               # residual connection ( I used summation)

        x5 = self.conv5(x4)
        x5 = self.pool(x5)
        x5 = x5.view(x5.size(0), -1)
        out = self.fc(x5)
        return out

# Set up device, model, loss, optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_res = PotatoCNNResidual(num_classes=3).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_res.parameters(), lr=1e-3)

# Retrain with residual connection
num_epochs = 10
train_losses_res= []
val_losses_res = []
train_acc_res= []
val_acc_res = []

for epoch in range(num_epochs):
    tr_loss, tr_acc = train_one_epoch(model_res, train_loader, criterion, optimizer, device)
    va_loss, va_acc = validate(model_res, test_loader, criterion, device)

    train_losses_res.append(tr_loss)
    val_losses_res.append(va_loss)
    train_acc_res.append(tr_acc)
    val_acc_res.append(va_acc)

    print(f"Epoch {epoch+1}/{num_epochs}: "
          f"Train Loss={tr_loss:.4f}, Train Acc={tr_acc:.2f}%, "
          f"Val Loss={va_loss:.4f}, Val Acc={va_acc:.2f}%")

#Plot the training and validation losses and training and validation accuracy
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses_res, label="Train Loss (Res)", marker="o")
plt.plot(range(1, num_epochs+1), val_losses_res, label="Val Loss (Res)", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Residual Model: Training and Validation Loss")
plt.legend()


plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_acc_res, label="Train Acc (Res)", marker="o")
plt.plot(range(1, num_epochs+1), val_acc_res, label="Val Acc (Res)", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Residual Model: Training and Validation Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

